# <center> Homework 136

In [1]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds
from pathlib import Path
import numpy as np

## Task 1
да се тестват примера в книгата с използване на готов модел за sentiment analysis

    да се пробват техниките от ползване на готови модели за класификация на изображения: заключване на горните слоеве при обучение, малък learning rate и т.н.


In [ ]:
raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised=True
)

tf.random.set_seed(42)
train_set = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = raw_valid_set.batch(32).prefetch(1)
test_set = raw_test_set.batch(32).prefetch(1)

In [ ]:
os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"

In [ ]:
url = "https://tfhub.dev/google/universal-sentence-encoder/4"

pretrain = hub.KerasLayer(url, trainable=False, dtype=tf.string, input_shape=[]),

In [ ]:
pretrain

In [ ]:
model_pretrain = tf.keras.Sequential([
    pretrain,
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x7c145bd4eba0> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)

In [ ]:
class PretarinModel(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.model = pretrain

    def call(self, inputs, training=False):
        return self.model(inputs, training=training)

inputs = tf.keras.Input(shape=[], dtype=tf.string)
x = PretarinModel()(inputs)
x = tf.keras.layers.Dense(64, activation="relu")(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_pretrain = tf.keras.Model(inputs, outputs)

### Warmup pretrain model trainable = False; learning rate = 0.01

In [ ]:
model_pretrain.compile(loss="binary_crossentropy",
                       optimizer=tf.keras.optimizers.Nadam(0.01),
                       metrics=["accuracy"])

model_pretrain.fit(train_set, validation_data=valid_set, epochs=3)

Epoch 1/3


704/704 ━━━━━━━━━━━━━━━━━━━━ 57s 73ms/step - accuracy: 0.8468 - loss: 0.3498 - val_accuracy: 0.8472 - val_loss: 0.3254
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 51s 72ms/step - accuracy: 0.8618 - loss: 0.3233 - val_accuracy: 0.8552 - val_loss: 0.3186
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 48s 69ms/step - accuracy: 0.8684 - loss: 0.3128 - val_accuracy: 0.8532 - val_loss: 0.3308


### Fine Tuning pretrain model trainable = True; learning rate = 0.0001

In [ ]:
model_pretrain.layers

[<InputLayer name=input_layer_3, built=True>,
 <PretarinModel name=pretarin_model_2, built=True>,
 <Dense name=dense_8, built=True>,
 <Dense name=dense_9, built=True>]

In [ ]:
model_pretrain.layers[1].model.trainable

False

In [ ]:
model_pretrain.layers[1].model.trainable = True
model_pretrain.layers[1].model.trainable

True

In [ ]:
model_pretrain.compile(loss="binary_crossentropy",
                       optimizer=tf.keras.optimizers.Nadam(0.0001),
                       metrics=["accuracy"])

callbacks = [tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True)]
model_pretrain.fit(train_set, validation_data=valid_set, epochs=20, callbacks=callbacks)

Epoch 1/20


704/704 ━━━━━━━━━━━━━━━━━━━━ 52s 69ms/step - accuracy: 0.8772 - loss: 0.2910 - val_accuracy: 0.8572 - val_loss: 0.3180
Epoch 2/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 49s 69ms/step - accuracy: 0.8809 - loss: 0.2859 - val_accuracy: 0.8580 - val_loss: 0.3157
Epoch 3/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 44s 62ms/step - accuracy: 0.8832 - loss: 0.2833 - val_accuracy: 0.8604 - val_loss: 0.3145
Epoch 4/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 39s 55ms/step - accuracy: 0.8837 - loss: 0.2815 - val_accuracy: 0.8624 - val_loss: 0.3142
Epoch 5/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 38s 54ms/step - accuracy: 0.8842 - loss: 0.2803 - val_accuracy: 0.8628 - val_loss: 0.3135
Epoch 6/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 38s 54ms/step - accuracy: 0.8849 - loss: 0.2793 - val_accuracy: 0.8620 - val_loss: 0.3131
Epoch 7/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.8859 - loss: 0.2784 - val_accuracy: 0.8636 - val_loss: 0.3126
Epoch 8/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 40s 56ms/step - accuracy: 0.8860 - loss: 0.2775 - val_accurac

In [ ]:
model_pretrain.evaluate(test_set)

782/782 ━━━━━━━━━━━━━━━━━━━━ 46s 58ms/step - accuracy: 0.8609 - loss: 0.3173


[0.3172881603240967, 0.8608800172805786]

## Task 2
да се имплементира и тества encoder-decoder модела от книгата

In [2]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets/spa-eng", extract=True)

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
path

'/tmp/.keras/datasets/spa-eng_extracted'

In [3]:
# text = (Path('datasets/spa-eng/datasets/spa-eng_extracted/spa-eng').with_name("spa-eng") / "spa.txt").read_text()
text = (Path('/tmp/.keras/datasets/spa-eng_extracted/spa-eng').with_name("spa-eng") / "spa.txt").read_text()

In [4]:
print(text[:50])

Go.	Ve.
Go.	Vete.
Go.	Vaya.
Go.	Váyase.
Hi.	Hola.



In [5]:
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs)

In [6]:
for i in range(3):
    print(sentences_en[i], "=>", sentences_es[i])

She met him on the beach. => Ella lo conoció en la playa.
I have a very vague idea of what you are talking about. => Tengo una idea muy vaga de lo que estás hablando.
Tomorrow is the last day of school! => Mañana es el último día de clase!


In [7]:
vocab_size = 1000
max_length = 50

text_vec_layer_en = tf.keras.layers.TextVectorization(
                        vocab_size, output_sequence_length=max_length)

text_vec_layer_es = tf.keras.layers.TextVectorization(
                        vocab_size, output_sequence_length=max_length)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

In [ ]:
print(text_vec_layer_en.get_vocabulary()[:10])

['', '[UNK]', np.str_('the'), np.str_('i'), np.str_('to'), np.str_('you'), np.str_('tom'), np.str_('a'), np.str_('is'), np.str_('he')]


In [ ]:
print(text_vec_layer_es.get_vocabulary()[:10])

['', '[UNK]', np.str_('startofseq'), np.str_('endofseq'), np.str_('de'), np.str_('que'), np.str_('a'), np.str_('no'), np.str_('tom'), np.str_('la')]


In [ ]:
sentences_en[:2]

('The U.N. building is very impressive.', 'Nice seeing you!')

In [8]:
X_train = tf.constant(sentences_en[:100_000])
X_valid = tf.constant(sentences_en[100_000:])

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])

In [9]:
Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])

In [ ]:
X_train[:2]

<tf.Tensor: shape=(2,), dtype=string, numpy=
array([b'The U.N. building is very impressive.', b'Nice seeing you!'],
      dtype=object)>

In [ ]:
X_train.shape

TensorShape([100000])

In [ ]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.LSTM(512, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

# DECODER
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

# OUTPUT
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(decoder_outputs)

nmt_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
nmt_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

callbacks = [tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/ml_models/nmt_model.keras')]
nmt_model.fit((X_train, X_train_dec), Y_train,
              epochs=10,
              validation_data=((X_valid, X_valid_dec), Y_valid),
              callbacks=callbacks)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 84s 24ms/step - accuracy: 0.0505 - loss: 3.5440 - val_accuracy: 0.0730 - val_loss: 2.2409
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 76s 24ms/step - accuracy: 0.0770 - loss: 2.0166 - val_accuracy: 0.0862 - val_loss: 1.6883
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 84s 24ms/step - accuracy: 0.0901 - loss: 1.4935 - val_accuracy: 0.0927 - val_loss: 1.4507
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.0979 - loss: 1.2130 - val_accuracy: 0.0958 - val_loss: 1.3457
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 76s 24ms/step - accuracy: 0.1035 - loss: 1.0281 - val_accuracy: 0.0968 - val_loss: 1.3026
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 93s 28ms/step - accuracy: 0.1078 - loss: 0.8875 - val_accuracy: 0.0973 - val_loss: 1.2982
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 76s 24ms/step - accuracy: 0.1118 - loss: 0.7650 - val_accuracy: 0.0975 - val_loss: 1.3047
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 25ms/step - accuracy: 0.1154 -

In [19]:
def translate(sentence_en):
    translation = ""
    for word_idx in range(max_length):
        X = tf.constant([sentence_en])  # encoder input
        X_dec = tf.constant(["startofseq " + translation])  # decoder input
        y_proba = nmt_model.predict((X, X_dec))[0, word_idx]  # last token's probas
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()

In [ ]:
translate("I like soccer")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


'me gusta el fútbol'

In [ ]:
translate("I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


'me gusta el tenis y no quiero ir a la playa'

## Task 3
да се обучи същия модел със тази стратегия за подаване на данни на декодера:

    gradually switching from feeding the decoder the previous target token to feeding it the previous output token during training.

да се сравнят резлтатите на двата модела.

In [15]:
class CustomRNN(tf.keras.layers.Layer):
    def __init__(self, cell, return_sequences=False, **kwargs):
        super().__init__(**kwargs)

        self.cell = cell
        self.return_sequences = return_sequences
        self.teacher_forcing_ratio = 1.
        self.supports_masking = True
        self.dense = None
        self.emb   = None

    def use_prediction(self, out, state, training):
        proba = self.dense(out)
        inxs = tf.argmax(proba, axis=1)
        emb_out = self.emb(inxs)
        return self.cell(emb_out, state, training=training)

    def call(self, inputs, mask=None, initial_state=None, training=False):
        batch, time_steps = tf.shape(inputs)[0], inputs.shape[1]

        ta = tf.TensorArray(inputs.dtype, time_steps)

        hd_states_init = self.cell.state_size
        if not isinstance(hd_states_init, (list, tuple)):
            hd_states_init = [hd_states_init]

        state = initial_state
        if state is None:
            state = [tf.zeros([batch, size], dtype=inputs.dtype) for size in hd_states_init]

        out = tf.zeros([batch, self.cell.output_size], dtype=inputs.dtype)

        for time_st in range(time_steps):
            time_inp = inputs[:, time_st]

            rnd = tf.random.uniform([], minval=0, maxval=1)

            use_prediction = tf.logical_and(
                rnd > self.teacher_forcing_ratio,
                time_st != 0
            )

            new_out, new_state = tf.cond(
                tf.cast(use_prediction, tf.bool),
                lambda: self.use_prediction(out, state, training),
                lambda: self.cell(time_inp, state, training=training)
            )

            if mask is not None:
                time_mask = tf.cast(mask[:, time_st:time_st+1], inputs.dtype)
                out = new_out * time_mask + out * (1 - time_mask)
                state = [new * time_mask + old * (1 - time_mask) for new, old in zip(new_state, state)]
            else:
                state = new_state
                out = new_out

            if self.return_sequences:
                ta = ta.write(time_st, out)

        output = out
        if self.return_sequences:
            output = ta.stack()
            output = tf.transpose(output, [1, 0, 2])

        if mask is None or not self.return_sequences:
            return output

        return output, self.compute_mask(inputs, mask)

    def get_config(self):
        config = super().get_config()
        config.update({
            "return_sequences": self.return_sequences,
            "teacher_forcing_ratio": self.teacher_forcing_ratio,
            "cell" : tf.keras.layers.serialize(self.cell),
            'dense': tf.keras.layers.serialize(self.dense),
            'emb'  : tf.keras.layers.serialize(self.emb)
        })
        return config

class TeacherForceScheduler(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        new_ratio = self.model.decoder.teacher_forcing_ratio * 0.9
        self.model.decoder.teacher_forcing_ratio = max(0.2, new_ratio)

In [12]:
x = 1

for _ in range(10):
    x *= 0.9
    print(x)

0.9
0.81
0.7290000000000001
0.6561000000000001
0.5904900000000002
0.5314410000000002
0.47829690000000014
0.43046721000000016
0.38742048900000015
0.34867844010000015


In [16]:
embed_size = 128

# INPUTS
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# VECTORIZATION
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# EMBEDDING
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

# ENCODER
encoder = tf.keras.layers.LSTM(512, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

# DECODER & OUTPUT
decoder = CustomRNN(tf.keras.layers.LSTMCell(512), return_sequences=True)
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")

decoder.dense = output_layer
decoder.emb = decoder_embedding_layer

decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)
Y_proba = output_layer(decoder_outputs)

nmt_model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], outputs=[Y_proba])
nmt_model.decoder = decoder

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'lstm_cell_1' (of type LSTMCell) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [17]:
nmt_model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

callbacks = [TeacherForceScheduler(),
             tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/ml_models/nmt_model_noteach.keras')]
nmt_model.fit((X_train, X_train_dec), Y_train,
              epochs=10,
              validation_data=((X_valid, X_valid_dec), Y_valid),
              callbacks=callbacks)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 208s 52ms/step - accuracy: 0.0509 - loss: 3.5153 - val_accuracy: 0.0746 - val_loss: 2.1596
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 154s 49ms/step - accuracy: 0.0789 - loss: 1.9456 - val_accuracy: 0.0875 - val_loss: 1.6379
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 202s 49ms/step - accuracy: 0.0916 - loss: 1.4388 - val_accuracy: 0.0931 - val_loss: 1.4256
Epoch 4/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 155s 49ms/step - accuracy: 0.0991 - loss: 1.1823 - val_accuracy: 0.0953 - val_loss: 1.3393
Epoch 5/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 155s 50ms/step - accuracy: 0.1046 - loss: 1.0010 - val_accuracy: 0.0969 - val_loss: 1.2988
Epoch 6/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 155s 50ms/step - accuracy: 0.1088 - loss: 0.8622 - val_accuracy: 0.0970 - val_loss: 1.3003
Epoch 7/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 153s 49ms/step - accuracy: 0.1125 - loss: 0.7451 - val_accuracy: 0.0970 - val_loss: 1.3177
Epoch 8/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 156s 50ms/step - accuracy: 

In [18]:
nmt_model.decoder.teacher_forcing_ratio

0.34867844010000015

In [20]:
translate("I like soccer")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'lstm_cell_1' (of type LSTMCell) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step


'me gusta jugar al tenis'

In [21]:
translate("I like soccer and also going to the beach")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step


'me gusta jugar a las cartas y a la vez'

In [23]:
translate("I love watching movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


'me encanta ver la película'